In [1]:
import os
import matplotlib.pyplot as plt
import pandas as pd
from ultralytics import YOLO

DATA_YAML        = 'data/detection.yaml'
PRETRAINED_SMALL = 'yolov8s.pt'
PROJECT_NAME  = 'kidney_stone_detector_small'
TRAIN_RUN_DIR = os.path.join('runs', PROJECT_NAME)
RESULTS_CSV   = os.path.join(TRAIN_RUN_DIR, 'results.csv')
WEIGHTS_PATH  = os.path.join(TRAIN_RUN_DIR, 'weights', 'last.pt')
BESt_WEIGHTS_PATH=os.path.join(TRAIN_RUN_DIR, 'weights', 'best.pt')
EVAL_METRICS_FILE = os.path.join('eval_metrice.txt')

EPOCHS           = 30
IMG_SIZE         = 320
BATCH_SIZE       = 2

def train_model():
        model = YOLO(WEIGHTS_PATH)
        print(f"Training for {EPOCHS} epochs on {DATA_YAML}...")
        model.train(
            data=DATA_YAML,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            device='cpu',
            project='runs',
            name=PROJECT_NAME,
            resume=True
        )

def save_eval_metrics(metrics, save_path):
    prec = metrics.box.mp
    rec  = metrics.box.mr
    with open(save_path, 'w') as f:
        f.write("Evaluation Metrics (YOLOv8-small):\n")
        f.write(f"Precision: {prec:.4f}\n")
        f.write(f"Recall:    {rec:.4f}\n")
        f.write(f"mAP@0.5:   {metrics.box.map50:.4f}\n")

def evaluate_model():
    model = YOLO(BESt_WEIGHTS_PATH)
    print("\nRunning evaluation on test set...")
    metrics = model.val(data=DATA_YAML, split='test')
    print(f" Precision: {metrics.box.mp:.4f}")
    print(f" Recall:    {metrics.box.mr:.4f}")
    print(f" mAP@0.5:   {metrics.box.map50:.4f}")
    save_eval_metrics(metrics, EVAL_METRICS_FILE)

def load_eval_metrics(file_path):
    if os.path.exists(file_path):
        print("\nLoaded saved evaluation metrics:")
        print(open(file_path).read())
    else:
        print(f"\nNo saved evaluation metrics found at {file_path}")

def show_training_metrics(csv_path):
    if not os.path.exists(csv_path):
        print(f"\n⚠Training metrics file not found at {csv_path}")
        return
    df   = pd.read_csv(csv_path)
    last = df.iloc[-1]
    print("\n=== Training Summary (YOLOv8-small, Last Epoch) ===")
    print(f"Epoch:     {int(last['epoch'])}")
    print(f"Precision: {last['metrics/precision(B)']:.4f}")
    print(f"Recall:    {last['metrics/recall(B)']:.4f}")
    print(f"mAP@0.5:   {last['metrics/mAP50(B)']:.4f}")

def test_image(img_path):
    model = YOLO(BESt_WEIGHTS_PATH)
    results = model.predict(
        source=img_path,
        imgsz=IMG_SIZE,
    )
    res = results[0]
    annotated = res.plot()
    plt.figure(figsize=(6,6))
    plt.imshow(annotated)
    plt.axis('off')
    plt.show()

    confs = res.boxes.conf.cpu().numpy()
    if len(confs) == 0:
        print("No kidney stone detected.")
    else:
        print("Kidney stone(s) detected!")
        for i, c in enumerate(confs, 1):
            print(f"  - Detection {i}: {c:.2%} confidence")


#train_model()

Training for 30 epochs on data/detection.yaml...
Ultralytics 8.3.138  Python-3.11.8 torch-2.1.2+cpu CPU (13th Gen Intel Core(TM) i5-1335U)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/detection.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=runs\kidney_stone_detector_small\weights\last.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=kidney_stone_detector_small, nbs=64, nms=False, opset=None, optimize=F

train: Scanning C:\Users\anasw\PycharmProjects\PythonProject1\KidneyStone1\data\train\labels.cache... 3382 images, 5 backgrounds, 297 corrupt: 100%|██████████| 3382/3382 [00:00<?, ?it/s]

train: C:\Users\anasw\PycharmProjects\PythonProject1\KidneyStone1\data\train\images\normal-kidney-10-_jpg.rf.49a65f555dcbf1297691d90bca24553b.jpg: ignoring corrupt image/label: Label class 1 exceeds dataset class count 1. Possible class labels are 0-0
train: C:\Users\anasw\PycharmProjects\PythonProject1\KidneyStone1\data\train\images\normal-kidney-10-_jpg.rf.677c02ecf516d9b9d979c5f6cc98f237.jpg: ignoring corrupt image/label: Label class 1 exceeds dataset class count 1. Possible class labels are 0-0
train: C:\Users\anasw\PycharmProjects\PythonProject1\KidneyStone1\data\train\images\normal-kidney-10-_jpg.rf.d1f95d1bc896603eb39f01c49b809acd.jpg: ignoring corrupt image/label: Label class 1 exceeds dataset class count 1. Possible class labels are 0-0
train: C:\Users\anasw\PycharmProjects\PythonProject1\KidneyStone1\data\train\images\normal-kidney-100-_jpg.rf.88e1aa5cab8829c5ae22c6d0d7b8305d.jpg: ignoring corrupt image/label: Label class 1 exceeds dataset class count 1. Possible class labels


val: Scanning C:\Users\anasw\PycharmProjects\PythonProject1\KidneyStone1\data\val\labels.cache... 144 images, 0 backgrounds, 14 corrupt: 100%|██████████| 144/144 [00:00<?, ?it/s]

val: C:\Users\anasw\PycharmProjects\PythonProject1\KidneyStone1\data\val\images\normal-kidney-1-_jpg.rf.0eac0c53c551f5d72816b996785e9e90.jpg: ignoring corrupt image/label: Label class 1 exceeds dataset class count 1. Possible class labels are 0-0
val: C:\Users\anasw\PycharmProjects\PythonProject1\KidneyStone1\data\val\images\normal-kidney-104-_jpg.rf.cb6d2338842b8fc17768de958e528f1e.jpg: ignoring corrupt image/label: Label class 1 exceeds dataset class count 1. Possible class labels are 0-0
val: C:\Users\anasw\PycharmProjects\PythonProject1\KidneyStone1\data\val\images\normal-kidney-110-_jpg.rf.2538382e2470b65701b52145e1d7183e.jpg: ignoring corrupt image/label: Label class 1 exceeds dataset class count 1. Possible class labels are 0-0
val: C:\Users\anasw\PycharmProjects\PythonProject1\KidneyStone1\data\val\images\normal-kidney-12-_jpg.rf.c23dc8b8cd347dcf9f50a1406a7b8b7c.jpg: ignoring corrupt image/label: Label class 1 exceeds dataset class count 1. Possible class labels are 0-0
val: C:

...

                   all        130        281      0.758      0.584      0.653      0.267



6 epochs completed in 1.526 hours.
Optimizer stripped from runs\kidney_stone_detector_small\weights\last.pt, 22.5MB
Optimizer stripped from runs\kidney_stone_detector_small\weights\best.pt, 22.5MB

Validating runs\kidney_stone_detector_small\weights\best.pt...
Ultralytics 8.3.138  Python-3.11.8 torch-2.1.2+cpu CPU (13th Gen Intel Core(TM) i5-1335U)
Model summary (fused): 72 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 33/33 [00:21<00:00,  1.53it/s]


                   all        130        281      0.755      0.584      0.653      0.267
Speed: 0.6ms preprocess, 156.3ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to runs\kidney_stone_detector_small


<Figure size 640x480 with 1 Axes>

In [2]:
evaluate_model()


Running evaluation on test set...
Ultralytics 8.3.138  Python-3.11.8 torch-2.1.2+cpu CPU (13th Gen Intel Core(TM) i5-1335U)
Model summary (fused): 72 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 87.85.2 MB/s, size: 33.1 KB)


val: Scanning C:\Users\anasw\PycharmProjects\PythonProject1\KidneyStone1\data\test\labels... 141 images, 0 backgrounds, 11 corrupt: 100%|██████████| 141/141 [00:00<00:00, 413.42it/s]

val: C:\Users\anasw\PycharmProjects\PythonProject1\KidneyStone1\data\test\images\normal-kidney-109-_jpg.rf.509e720ce622cb94262363ea1c118bb9.jpg: ignoring corrupt image/label: Label class 1 exceeds dataset class count 1. Possible class labels are 0-0
val: C:\Users\anasw\PycharmProjects\PythonProject1\KidneyStone1\data\test\images\normal-kidney-127-_jpg.rf.9fd750bfc522ea277ef15a8144cd13b8.jpg: ignoring corrupt image/label: Label class 1 exceeds dataset class count 1. Possible class labels are 0-0
val: C:\Users\anasw\PycharmProjects\PythonProject1\KidneyStone1\data\test\images\normal-kidney-20-_jpg.rf.eed51472a44091707622ed11496188f7.jpg: ignoring corrupt image/label: Label class 1 exceeds dataset class count 1. Possible class labels are 0-0
val: C:\Users\anasw\PycharmProjects\PythonProject1\KidneyStone1\data\test\images\normal-kidney-22-_jpg.rf.8c0d4b5f129e992254dce01cc8522900.jpg: ignoring corrupt image/label: Label class 1 exceeds dataset class count 1. Possible class labels are 0-0
va


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 9/9 [00:09<00:00,  1.01s/it]


                   all        130        265      0.766      0.619      0.667      0.274
Speed: 0.3ms preprocess, 63.2ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to runs\detect\val
 Precision: 0.7657
 Recall:    0.6189
 mAP@0.5:   0.6670


In [3]:
show_training_metrics(RESULTS_CSV)
load_eval_metrics(EVAL_METRICS_FILE)


=== Training Summary (YOLOv8-small, Last Epoch) ===
Epoch:     30
Precision: 0.7579
Recall:    0.5836
mAP@0.5:   0.6531

Loaded saved evaluation metrics:
Evaluation Metrics (YOLOv8-small):
Precision: 0.7657
Recall:    0.6189
mAP@0.5:   0.6670



In [5]:
img_path = input("\nEnter path to your image for stone detection: ")
test_image(img_path)


image 1/1 C:\Users\anasw\OneDrive\Desktop\ANN PROJECT\Unseen CT images\stone\Stone- (1152).jpg: 320x320 2 stones, 100.4ms
Speed: 2.3ms preprocess, 100.4ms inference, 0.7ms postprocess per image at shape (1, 3, 320, 320)


<Figure size 600x600 with 1 Axes>

Kidney stone(s) detected!
  - Detection 1: 61.79% confidence
  - Detection 2: 50.40% confidence
